In [10]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd

SCRATCH = Path("/n/netscratch/eddy_lab/Everyone/nhilgert")
SENT = SCRATCH / "2026_11_06_qc_regen_nick_andy_coords"           # objects sent to Cagri, Nov 6 (the record)
RERUN = SCRATCH / "2026_11_06_xmens_preprocess" / "with_coords"    # the re-run from FASTQs, Nov 8

In [11]:
def same_sparse(a, b):
    """True when two sparse matrices hold identical values at identical positions."""
    return a.shape == b.shape and (a != b).nnz == 0

def compare(species):
    sent = ad.read_h5ad(SENT / f"qc_passed_{species}.h5ad")
    rerun = ad.read_h5ad(RERUN / f"qc_passed_{species}.h5ad")
    print(f"=== {species}: sent {sent.shape}, re-run {rerun.shape}")

    checks = {
        "nucleus IDs, same order": sent.obs_names.equals(rerun.obs_names),
        "gene IDs, same order": sent.var_names.equals(rerun.var_names),
        "X (counts) identical": same_sparse(sent.X, rerun.X),
        "layers['counts'] identical": same_sparse(sent.layers["counts"], rerun.layers["counts"]),
        "obs columns, same set": set(sent.obs.columns) == set(rerun.obs.columns),
        "obsm keys, same set": set(sent.obsm) == set(rerun.obsm),
        "var table identical": sent.var.equals(rerun.var),
    }
    # obs: column by column, NaN == NaN (unplaced nuclei have NaN positions)
    for col in sent.obs.columns:
        a, b = sent.obs[col], rerun.obs[col]
        if pd.api.types.is_numeric_dtype(a):
            checks[f"obs['{col}'] identical"] = np.array_equal(a.to_numpy(float), b.to_numpy(float), equal_nan=True)
        else:
            checks[f"obs['{col}'] identical"] = a.astype(str).equals(b.astype(str))
    for key in sent.obsm:
        checks[f"obsm['{key}'] identical"] = np.array_equal(sent.obsm[key], rerun.obsm[key], equal_nan=True)

    for name, ok in checks.items():
        print(f"  {'OK ' if ok else 'DIFF'}  {name}")

    # uns: the one known difference is the wording of a provenance note
    print("  uns keys:", list(sent.uns), "|", list(rerun.uns))
    for key, sent_text in sent.uns["coordinate_sources"].items():
        rerun_text = rerun.uns["coordinate_sources"][key]
        if sent_text == rerun_text:
            print(f"  OK    uns['coordinate_sources'][{key!r}] identical")
        else:
            print(f"  DIFF  uns['coordinate_sources'][{key!r}]\n          sent:   {sent_text}\n          re-run: {rerun_text}")

    n_placed = int(sent.obs["x_um_dbscan_nick"].notna().sum())
    print(f"  {sent.n_obs:,} nuclei compared, {n_placed:,} with a Nick position, "
          f"{int(sent.obs['x_um_dbscan_andy'].notna().sum()):,} with an Andy position")
    return all(checks.values())

In [12]:
all_mouse = compare("mouse")
all_human = compare("human")
print("\nevery check OK except the provenance wording:", all_mouse and all_human)

=== mouse: sent (69307, 32287), re-run (69307, 32287)
  OK   nucleus IDs, same order
  OK   gene IDs, same order
  OK   X (counts) identical
  OK   layers['counts'] identical
  OK   obs columns, same set
  OK   obsm keys, same set
  OK   var table identical
  OK   obs['sample'] identical
  OK   obs['species'] identical
  OK   obs['timepoint'] identical
  OK   obs['x_um_dbscan'] identical
  OK   obs['y_um_dbscan'] identical
  OK   obs['n_genes_by_counts'] identical
  OK   obs['total_counts'] identical
  OK   obs['pct_counts_mt'] identical
  OK   obs['pct_counts_ribo'] identical
  OK   obs['doubletdetection_predicted_doublet'] identical
  OK   obs['doubletdetection_doublet_score'] identical
  OK   obs['cell_barcode'] identical
  OK   obs['x_um_dbscan_nick'] identical
  OK   obs['y_um_dbscan_nick'] identical
  OK   obs['x_um_dbscan_andy'] identical
  OK   obs['y_um_dbscan_andy'] identical
  OK   obsm['spatial'] identical
  OK   obsm['spatial_andy'] identical
  uns keys: ['coordinate_sourc